In [0]:
from pyspark.sql import functions as F
from functools import reduce

CATALOG, Y = "workspace", 2025
b = spark.table(f"{CATALOG}.rc_bronze.survey_results").where(f"_survey_year = {Y}")
src_cols = [r.bronze_name for r in spark.table(f"{CATALOG}.rc_bronze.column_map")
            .where(f"survey_year = {Y}").orderBy("ordinal").collect()]

aggs = []
for c in src_cols:
    aggs += [F.sum((F.col(c) != "NA").cast("int")).alias(f"{c}__n"),
             F.sum(F.expr(f"try_cast({c} as double) is not null").cast("int")).alias(f"{c}__num")]
r = b.agg(*aggs).collect()[0].asDict()
numeric_cols = [c for c in src_cols if r[f"{c}__n"] > 0 and r[f"{c}__n"] == r[f"{c}__num"]]
print(f"Colonnes entièrement numériques ({len(numeric_cols)}) :")
print(numeric_cols)

Colonnes entièrement numériques (53) :
['ResponseId', 'WorkExp', 'YearsCode', 'TechEndorse_1', 'TechEndorse_2', 'TechEndorse_3', 'TechEndorse_4', 'TechEndorse_5', 'TechEndorse_6', 'TechEndorse_7', 'TechEndorse_8', 'TechEndorse_9', 'TechEndorse_13', 'TechOppose_1', 'TechOppose_2', 'TechOppose_3', 'TechOppose_5', 'TechOppose_7', 'TechOppose_9', 'TechOppose_11', 'TechOppose_13', 'TechOppose_16', 'TechOppose_15', 'JobSatPoints_1', 'JobSatPoints_2', 'JobSatPoints_3', 'JobSatPoints_4', 'JobSatPoints_5', 'JobSatPoints_6', 'JobSatPoints_7', 'JobSatPoints_8', 'JobSatPoints_9', 'JobSatPoints_10', 'JobSatPoints_11', 'JobSatPoints_13', 'JobSatPoints_14', 'JobSatPoints_15', 'JobSatPoints_16', 'ToolCountWork', 'ToolCountPersonal', 'CompTotal', 'SO_Actions_1', 'SO_Actions_16', 'SO_Actions_3', 'SO_Actions_4', 'SO_Actions_5', 'SO_Actions_6', 'SO_Actions_9', 'SO_Actions_7', 'SO_Actions_10', 'SO_Actions_15', 'ConvertedCompYearly', 'JobSat']


In [0]:
s = b.agg(*[F.sum(F.col(c).contains(";").cast("int")).alias(c) for c in src_cols]).collect()[0].asDict()
semi_cols = [c for c in src_cols if (s[c] or 0) >= 100]
print(f"Colonnes avec ';' sur au moins 100 lignes : {len(semi_cols)}")

rows = []
for c in semi_cols:
    n_items = (b.where(F.col(c) != "NA")
                 .select(F.explode(F.split(c, ";")).alias("item"))
                 .select("item").distinct().count())
    rows.append((c, int(s[c]), n_items))
spark.createDataFrame(rows, "colonne string, lignes_avec_pv long, elements_distincts long") \
     .orderBy("elements_distincts").show(80, truncate=False)

Colonnes avec ';' sur au moins 100 lignes : 58
+----------------------------------------+--------------+------------------+
|colonne                                 |lignes_avec_pv|elements_distincts|
+----------------------------------------+--------------+------------------+
|AIFrustration                           |17956         |7                 |
|AIAgentChallengesNeutral                |12399         |7                 |
|AIAgentChallengesSomewhat_disagree      |4169          |7                 |
|AIAgentChallengesStrongly_agree         |16315         |7                 |
|AIAgentChallengesSomewhat_agree         |13478         |7                 |
|AIAgentChallengesStrongly_disagree      |2259          |7                 |
|AgentUsesGeneral                        |3016          |8                 |
|AIAgentImpactSomewhat_agree             |7975          |8                 |
|AIAgentImpactNeutral                    |6132          |8                 |
|AIAgentImpactSomewhat_disagr

In [0]:
lvl = [c for c in src_cols if c.startswith("AIAgentChallenges")]
parts = [b.where(F.col(c) != "NA")
          .select("ResponseId", F.lit(c).alias("lvl"), F.explode(F.split(c, ";")).alias("item")) for c in lvl]
d = reduce(lambda x, y: x.unionByName(y), parts)
dup = d.groupBy("ResponseId", "item").count().where("count > 1").count()
print(f"AIAgentChallenges : {len(lvl)} colonnes | {d.select('ResponseId').distinct().count():,} répondants"
      f" | {d.count():,} paires | paires à plusieurs niveaux : {dup:,}")

AIAgentChallenges : 5 colonnes | 28,954 répondants | 174,136 paires | paires à plusieurs niveaux : 0


In [0]:
from pyspark.sql import functions as F
from functools import reduce
from collections import defaultdict

CATALOG, Y = "workspace", 2025
SILVER = f"{CATALOG}.rc_silver.survey_clean"
MULTI = f"{CATALOG}.rc_silver.survey_multi"
ANSWERED = f"{CATALOG}.rc_silver.survey_multi_answered"

FREE_TEXT = {"AIExplain", "AIOpen"}                       # textes libres : jamais découpés
MATRIX_PREFIXES = ["AIAgentImpact", "AIAgentChallenges", "AITool"]
TECH_SUFFIXES = ["HaveWorkedWith", "WantToWorkWith", "Admired"]
MIN_SEMI_ROWS = 100

s = spark.table(SILVER).where(f"_survey_year = {Y}")
dtypes = dict(s.dtypes)
src_cols = [r.bronze_name for r in spark.table(f"{CATALOG}.rc_bronze.column_map")
            .where(f"survey_year = {Y}").orderBy("ordinal").collect()]

semi = s.agg(*[F.sum(F.col(c).contains(";").cast("int")).alias(c)
               for c in src_cols if dtypes[c] == "string"]).collect()[0].asDict()

def classify(col):
    for p in MATRIX_PREFIXES:
        if col.startswith(p):
            return p, col[len(p):].lstrip("_")
    for suf in TECH_SUFFIXES:
        if col.endswith(suf):
            return col[: -len(suf)], suf
    return col, None

multi_cols = [c for c in src_cols
              if dtypes[c] == "string" and c not in FREE_TEXT
              and (any(c.startswith(p) for p in MATRIX_PREFIXES) or (semi.get(c) or 0) >= MIN_SEMI_ROWS)]
plan = [(c,) + classify(c) for c in multi_cols]
assert len(multi_cols) == 56, f"CRITIQUE : {len(multi_cols)} colonnes retenues au lieu de 56"
print(f"{len(multi_cols)} colonnes à normaliser, {len({g for _, g, _ in plan})} groupes de questions")

def explode_col(c, grp, lvl):
    return (s.where(F.col(c).isNotNull())
             .select("ResponseId",
                     F.lit(grp).alias("question_group"),
                     F.lit(c).alias("source_column"),
                     F.lit(lvl).cast("string").alias("level"),
                     F.explode(F.split(F.col(c), ";")).alias("raw_item")))

long_df = (reduce(lambda x, y: x.unionByName(y), [explode_col(c, g, l) for c, g, l in plan])
           .withColumn("item", F.trim("raw_item"))
           .withColumn("item_norm", F.regexp_replace("item", "[\u2018\u2019]", "'")))

chk = long_df.agg(F.count("*").alias("n"),
                  F.sum((F.col("item") == "").cast("int")).alias("vides"),
                  F.sum((F.col("raw_item") != F.col("item")).cast("int")).alias("espaces")).collect()[0]

# Méthode indépendante : 1 + nombre de ';' par cellule
exp = s.agg(*[F.sum(F.when(F.col(c).isNotNull(),
                           F.length(c) - F.length(F.regexp_replace(c, ";", "")) + 1).otherwise(0)).alias(c)
              for c in multi_cols]).collect()[0].asDict()
expected = sum(exp.values())

print(f"Lignes longues brutes : {chk['n']:,} | attendues (comptage des ';') : {expected:,}")
print(f"Éléments vides : {chk['vides']:,} | éléments avec espaces en trop : {chk['espaces']:,}")
assert chk["n"] == expected, "CRITIQUE : le découpage ne réconcilie pas avec le comptage des ';'"

56 colonnes à normaliser, 26 groupes de questions
Lignes longues brutes : 3,659,487 | attendues (comptage des ';') : 3,659,487
Éléments vides : 0 | éléments avec espaces en trop : 0


In [0]:
(long_df.where("item != ''")
   .select(F.lit(Y).alias("survey_year"), "ResponseId", "question_group", "source_column",
           "level", "item", "item_norm")
   .dropDuplicates(["ResponseId", "source_column", "item_norm"])
   .withColumn("_silver_processed_at", F.current_timestamp())
   .write.format("delta").mode("overwrite")
   .option("replaceWhere", f"survey_year = {Y}").saveAsTable(MULTI))

m = spark.table(MULTI).where(f"survey_year = {Y}")
print(f"survey_multi : {m.count():,} lignes (brutes : {chk['n']:,})")

(m.select("survey_year", "ResponseId", "question_group").distinct()
   .write.format("delta").mode("overwrite")
   .option("replaceWhere", f"survey_year = {Y}").saveAsTable(ANSWERED))
print("survey_multi_answered :", spark.table(ANSWERED).where(f"survey_year = {Y}").count(), "lignes")

survey_multi : 3,659,487 lignes (brutes : 3,659,487)
survey_multi_answered : 584326 lignes


In [0]:
m = spark.table(MULTI).where(f"survey_year = {Y}")
a = spark.table(ANSWERED).where(f"survey_year = {Y}")

# 1. Protection contre le double comptage : clé unique
d = m.groupBy("ResponseId", "source_column", "item_norm").count().where("count > 1").count()
print("Doublons (répondant, colonne, élément) :", d)
assert d == 0, "CRITIQUE : double comptage"

# 2. Intégrité référentielle
orphans = m.select("ResponseId").distinct().join(s.select("ResponseId"), "ResponseId", "left_anti").count()
print("ResponseId orphelins :", orphans)
assert orphans == 0, "CRITIQUE : ResponseId absent de survey_clean"

# 3. Répondants par groupe = méthode indépendante sur Silver
cols_by_group = defaultdict(list)
for c, g, l in plan:
    cols_by_group[g].append(c)
got = {r["question_group"]: r["count"] for r in a.groupBy("question_group").count().collect()}
bad = []
for g, cs in cols_by_group.items():
    cond = reduce(lambda x, y: x | y, [F.col(c).isNotNull() for c in cs])
    e = s.where(cond).count()
    if e != got.get(g, 0):
        bad.append((g, e, got.get(g, 0)))
print("Groupes dont les répondants ne réconcilient pas :", bad)
assert not bad, "CRITIQUE : réconciliation des répondants par groupe"

# 4. Contrôles croisés avec l'EDA (33 556 / 31 522 / 31 066 / 12 840 / 28 954 attendus)
a.where("question_group in ('LearnCode','AIFrustration','AITool','AIAgentImpact','AIAgentChallenges')") \
 .groupBy("question_group").count().show(truncate=False)

# 5. Fréquences des frustrations liées à l'IA (par répondant, jamais par ligne)
den = got["AIFrustration"]
(m.where("question_group = 'AIFrustration'").groupBy("item_norm")
  .agg(F.countDistinct("ResponseId").alias("n"))
  .withColumn("pct", F.round(F.col("n") / den * 100, 1))
  .orderBy(F.desc("n")).show(truncate=False))

Doublons (répondant, colonne, élément) : 0
ResponseId orphelins : 0
Groupes dont les répondants ne réconcilient pas : []
+-----------------+-----+
|question_group   |count|
+-----------------+-----+
|LearnCode        |33556|
|AITool           |31066|
|AIFrustration    |31522|
|AIAgentImpact    |12840|
|AIAgentChallenges|28954|
+-----------------+-----+

+----------------------------------------------------+-----+----+
|item_norm                                           |n    |pct |
+----------------------------------------------------+-----+----+
|AI solutions that are almost right, but not quite   |20806|66.0|
|Debugging AI-generated code is more time-consuming  |14262|45.2|
|I don't use AI tools regularly                      |7413 |23.5|
|I've become less confident in my own problem-solving|6323 |20.1|
|It's hard to understand how or why the code works   |5147 |16.3|
|Other (write in):                                   |3659 |11.6|
|I haven't encountered any problems               